# PROJECT: AOVE Price Predictor - Macro-Aware Deep Learning
=============================================================================
OBJECTIVE: 
Predict the weekly origin price of Extra Virgin Olive Oil (AOVE) in €/kg 
using a bimodal PyTorch LSTM architecture. The model fuses spatially aggregated 
high-frequency climate data with publish-shifted macroeconomic indicators.
Implements strict MLOps standards for data leakage prevention and checkpointing.

1. SPATIAL AGGREGATION (THE "GLOBAL MARKET" FIX)
-----------------------------------------------------------------------------
- Problem: The market has a single price, not a price per municipality.
- Solution: Climate data from all municipalities is collapsed into a single 
  weekly sequence via a Weighted Average, using productive surface area (ha) 
  as the weight. 
- Result: Highly dense, noise-free weekly rows representing the true climatic 
  stress on the overall productive capacity.

2. PREVENTING DATA LEAKAGE (THE "TIME MACHINE" FIX)
-----------------------------------------------------------------------------
- Low-frequency data (Stock, IPC) is aligned based on PUBLICATION DATE.
- A strict 15-day forward shift is applied to reports before merging via 
  'backward' fill to simulate real-world trading latency.
- Scalers (StandardScaler) are fitted ONLY on the training split to prevent 
  future variance from leaking into the training phase.

3. FEATURE CATALOGUE & STATIONARITY
-----------------------------------------------------------------------------
Target (Y):
- aove_price_eur_kg      [Float32] : Weekly origin price (€/kg).

High-Frequency Dynamic (LSTM Sequence - Weighted Spatial Average):
- rainfall_mm            [Float32] : Accumulated weekly precipitation (mm).
- temp_max_c             [Float32] : Absolute weekly maximum temperature (°C).
- water_deficit_mm       [Float32] : P - ETP weekly balance (mm).
- time_sin               [Float32] : sin(week * 2 * pi / 52).
- time_cos               [Float32] : cos(week * 2 * pi / 52).

Low-Frequency Macroeconomic (Fused post-LSTM - Converted to % Delta/Log):
- stock_delta_pct        [Float32] : Olive oil stock variation.
- surface_delta_pct      [Float32] : Productive surface area variation.
- ipc_monthly            [Float32] : Consumer Price Index (inflation context).
- diesel_price_eur       [Float32] : Agricultural diesel cost floor.

4. MLOPS & COMPUTATIONAL STANDARDS
-----------------------------------------------------------------------------
- OOP Architecture: Strict separation of ETL, Sequencing, and Modeling.
- Chronological Split: Train/Validation datasets are split chronologically.
- Checkpointing: Saves state_dict only when Validation Loss improves.
- Inverse Transform: Predicts scaled values but returns real-world €/kg.
- Data Types: Forced casting to np.float32 and torch.float32.

In [ ]:
# Activar fine-tunning
import sys
sys.argv = ['aove_predictor.py', '--finetune']

In [1]:
# Desactivar fine-tunning
import sys
sys.argv = ['aove_predictor.py']

In [4]:

"""
AOVE Macro-Aware Price Predictor - Production Engine v2
=============================================================================
Industrial-grade PyTorch pipeline for AOVE price regression.

All calculations strictly enforced in float32.
"""

import os
import argparse
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Dict
# Logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)

# ==============================================================================
# 1. ETL & SPATIAL AGGREGATION
# ==============================================================================
class MarketETLPipeline:
    """
    Handles spatial weighted averaging of climate data and time-shifted
    merging of macroeconomic indicators.
    """
    def __init__(self, publish_delay_days: int = 15) -> None:
        self.publish_delay_days = publish_delay_days

    def aggregate_climate(self, df_municipal: pd.DataFrame) -> pd.DataFrame:
        """
        Collapses municipal climate data into a single weekly sequence using
        productive surface area (ha) as the spatial weight.

        [FIX-2] Uses include_groups=False for pandas >= 2.2 compatibility.
        [FIX-5] week_num cast to plain float before np.sin/cos.
        """
        logger.info("Performing spatial weighted aggregation on climate data...")

        climate_cols = ['rainfall_mm', 'temp_max_c', 'water_deficit_mm']

        def weighted_avg(group: pd.DataFrame) -> pd.Series:
            weights = group['surface_ha'].values.astype(np.float64)
            total = weights.sum()
            if total == 0:
                return group[climate_cols].mean()
            w_avg = np.average(group[climate_cols].values, weights=weights, axis=0)
            return pd.Series(w_avg, index=climate_cols)

        # [FIX-2] include_groups=False prevents the group key from appearing
        # inside the function and resolves InconsistentIndexError in pandas 2.2+
        df_agg = (
            df_municipal
            .groupby('date')[climate_cols + ['surface_ha']]
            .apply(weighted_avg, include_groups=False)
            .reset_index()
        )

        # [FIX-5] Cast to plain Python float before trigonometric ops
        week_num = df_agg['date'].dt.isocalendar().week.astype(float)
        df_agg['time_sin'] = np.sin(week_num * 2 * np.pi / 52).astype(np.float32)
        df_agg['time_cos'] = np.cos(week_num * 2 * np.pi / 52).astype(np.float32)

        return df_agg.set_index('date')

    def align_macro_data(
        self,
        df_climate: pd.DataFrame,
        df_macro: pd.DataFrame,
        aove_lag_weeks: int = 1,
    ) -> pd.DataFrame:
        """
        Shifts macroeconomic data by publication delay and merges backwards.

        [FIX-3] surface_delta_pct now included in the macro merge.
                 aove_price_eur_kg is NOT merged as a raw feature to avoid
                 direct target leakage. Instead, an explicit lag column
                 (aove_lag_price) is built from the target series with a
                 controlled weekly offset so the model sees past prices
                 transparently.

        Parameters
        ----------
        aove_lag_weeks : int
            Number of weeks to lag the AOVE price before exposing it as a
            feature.  Default 1 (price known one week ago).  Set to 0 to
            disable the lag feature entirely.
        """
        logger.info(f"Applying {self.publish_delay_days}-day publication shift to macro data...")

        df_macro = df_macro.copy()
        df_macro['publish_date'] = (
            df_macro['reference_date'] + pd.DateOffset(days=self.publish_delay_days)
        )
        df_macro_shifted = df_macro.sort_values('publish_date').set_index('publish_date')

        # [FIX-3] All legitimate macro features — aove_price_eur_kg excluded here
        macro_feature_cols = [
            'stock_delta_pct',
            'surface_delta_pct',
            'ipc_monthly',
            'diesel_price_eur',
        ]

        df_climate = df_climate.sort_index()
        df_final = pd.merge_asof(
            left=df_climate,
            right=df_macro_shifted[macro_feature_cols + ['aove_price_eur_kg']],
            left_index=True,
            right_index=True,
            direction='backward',
        )

        # [FIX-3] Build explicit lag feature from target, then drop raw target.
        # The lag is created BEFORE the global ffill/fillna pass so we can fill
        # its leading NaN(s) with bfill — i.e. the oldest known price — instead
        # of 0.0, which would be a spurious outlier that distorts the scaler fit.
        if aove_lag_weeks > 0:
            df_final['aove_lag_price'] = (
                df_final['aove_price_eur_kg']
                .shift(aove_lag_weeks)
                .bfill()          # fills the first aove_lag_weeks NaNs with the
                                  # oldest observed price — semantically correct
                                  # and harmless to the scaler statistics
            )
        # Keep aove_price_eur_kg only as the target column; it will be separated
        # later in TemporalSequenceBuilder.

        # ffill propagates the last known macro value forward (handles monthly→weekly).
        # fillna(0.0) is the last-resort fallback only for columns that have NO
        # valid observation at all (e.g. surface_delta_pct before its series starts).
        # aove_lag_price is already fully populated at this point, so 0.0 never
        # touches it.
        df_final.ffill(inplace=True)
        df_final.fillna(0.0, inplace=True)

        return df_final.astype(np.float32)

# ==============================================================================
# 2. SEQUENCE GENERATOR (CHRONOLOGICAL SPLIT — LEAKAGE-FREE)
# ==============================================================================
class TemporalSequenceBuilder:
    """
    Transforms the 1D timeline into overlapping historical windows.

    [FIX-1] The sequence split index is derived from the *dataframe* split
            index so that both share exactly the same temporal boundary:

            split_seq_idx = df_split_idx - time_steps

            Rationale: sliding window i uses rows [i, i+time_steps) and
            predicts row i+time_steps.  The first window whose *target* falls
            inside the validation period is window number
            (df_split_idx - time_steps).  Every window before that index has
            its target in the training period — no leakage.

    [IMP-1] Sliding windows built with np.lib.stride_tricks.sliding_window_view
            (zero-copy view, no joblib overhead).
    """
    def __init__(self, time_steps: int = 104) -> None:
        self.time_steps = time_steps
        self.scaler_hf     = StandardScaler()
        self.scaler_macro  = StandardScaler()
        self.scaler_target = StandardScaler()

    def build_sequences_split(
        self,
        df_final: pd.DataFrame,
        hf_cols: list,
        macro_cols: list,
        target_col: str,
        train_ratio: float = 0.8,
    ) -> Dict[str, Tuple[np.ndarray, np.ndarray, np.ndarray]]:
        """
        Builds sliding-window sequences with a strictly chronological split.
        Scalers are fitted on training rows only.
        """
        logger.info("Building sequences with chronological train/val split...")

        n = len(df_final)
        df_split_idx = int(n * train_ratio)

        if df_split_idx <= self.time_steps:
            raise ValueError(
                f"Training set too short: {df_split_idx} rows < time_steps={self.time_steps}. "
                "Reduce time_steps or increase train_ratio."
            )

        # 1. Fit scalers ONLY on training rows
        df_train = df_final.iloc[:df_split_idx]
        self.scaler_hf.fit(df_train[hf_cols].values)
        self.scaler_macro.fit(df_train[macro_cols].values)
        self.scaler_target.fit(df_train[[target_col]].values)

        # 2. Transform the full dataset
        hf_scaled     = self.scaler_hf.transform(df_final[hf_cols].values).astype(np.float32)
        macro_scaled  = self.scaler_macro.transform(df_final[macro_cols].values).astype(np.float32)
        target_scaled = self.scaler_target.transform(df_final[[target_col]].values).astype(np.float32).ravel()

        # 3. [IMP-1] Zero-copy sliding window view
        #    sliding_window_view produces (n - time_steps + 1) windows; the last
        #    window [n-time_steps : n] has no target row → drop it with [:-1].
        #    Final shape: (n - time_steps, time_steps, n_hf_features)
        hf_windows = np.lib.stride_tricks.sliding_window_view(
            hf_scaled, window_shape=self.time_steps, axis=0
        )
        # shape after view: (n - time_steps + 1, n_features, time_steps)
        # → drop last window, transpose to (n-time_steps, time_steps, n_features)
        X_hf = hf_windows[:-1].transpose(0, 2, 1).copy()   # copy() makes it contiguous

        # Macro: take the snapshot at the last step of each window (i + time_steps - 1)
        X_macro = macro_scaled[self.time_steps - 1 : n - 1]   # shape (n-time_steps, n_macro)

        # Target: the row immediately after each window
        y = target_scaled[self.time_steps:]                    # shape (n-time_steps,)

        # 4. [FIX-1] Derive sequence split from dataframe split
        #    First window whose target falls in val: df_split_idx - time_steps
        split_seq_idx = df_split_idx - self.time_steps

        logger.info(
            f"Total windows: {len(X_hf)} | "
            f"Train: {split_seq_idx} | "
            f"Val: {len(X_hf) - split_seq_idx}"
        )

        target_dates = df_final.index[self.time_steps:]
        val_dates    = target_dates[split_seq_idx:]

        return {
            "train": (
                X_hf[:split_seq_idx],
                X_macro[:split_seq_idx],
                y[:split_seq_idx],
            ),
            "val": (
                X_hf[split_seq_idx:],
                X_macro[split_seq_idx:],
                y[split_seq_idx:],
            ),
            "val_dates": val_dates,
        }

    def inverse_transform_target(self, y_scaled: np.ndarray) -> np.ndarray:
        """[FIX-4] Converts scaled predictions back to real-world €/kg."""
        return self.scaler_target.inverse_transform(
            y_scaled.reshape(-1, 1)
        ).ravel().astype(np.float32)

# ==============================================================================
# 3. BIMODAL LSTM REGRESSION MODEL
# ==============================================================================
class AOVEPricePredictor(nn.Module):
    """
    Bimodal deep learning architecture.

    Stream A — LSTM over the high-frequency climate sequence.
    Stream B — macro snapshot injected post-LSTM at the fusion layer.
    Both streams are concatenated and passed through a fully-connected head.
    """
    def __init__(
        self,
        hf_input_dim: int,
        macro_input_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=hf_input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)

        fusion_dim = hidden_dim + macro_input_dim
        self.fc = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x_hf: torch.Tensor, x_macro: torch.Tensor) -> torch.Tensor:
        h0 = torch.zeros(
            self.num_layers, x_hf.size(0), self.hidden_dim,
            dtype=torch.float32, device=x_hf.device,
        )
        c0 = torch.zeros(
            self.num_layers, x_hf.size(0), self.hidden_dim,
            dtype=torch.float32, device=x_hf.device,
        )
        lstm_out, _ = self.lstm(x_hf, (h0, c0))
        h_last = self.dropout(lstm_out[:, -1, :])
        fused  = torch.cat([h_last, x_macro], dim=1)
        return self.fc(fused)

# ==============================================================================
# 4. MLOPS TRAINING ENGINE
# ==============================================================================
class AOVETrainer:
    """
    Training loop with HuberLoss, ReduceLROnPlateau scheduler, early stopping,
    and model checkpointing.

    [IMP-2] ReduceLROnPlateau halves the LR when val_loss plateaus for
            `lr_patience` consecutive epochs.
    [IMP-3] Early stopping terminates training when val_loss has not improved
            for `es_patience` consecutive epochs.
    [FIX-4] predict() applies inverse_transform so outputs are in €/kg.
    """
    def __init__(
        self,
        model: nn.Module,
        sequence_builder: TemporalSequenceBuilder,
        learning_rate: float = 1e-3,
        lr_patience: int = 10,
        es_patience: int = 20,
    ) -> None:
        self.model             = model.to(DEVICE)
        self.sequence_builder  = sequence_builder
        self.criterion         = nn.HuberLoss(delta=1.0)
        self.optimizer         = torch.optim.Adam(
            self.model.parameters(), lr=learning_rate, weight_decay=1e-5
        )
        # [IMP-2] Scheduler
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=lr_patience, factor=0.5
        )
        self.es_patience = es_patience
        self.history = {'train': [], 'val': []}

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------
    def train_and_validate(
        self,
        train_loader: DataLoader,
        val_loader:   DataLoader,
        epochs: int,
        save_path: str = "best_aove_model.pth",
    ) -> None:
        best_val_loss    = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            # ── Training ──────────────────────────────────────────────
            self.model.train()
            train_loss = 0.0
            for X_hf, X_macro, y in train_loader:
                X_hf   = X_hf.to(DEVICE,   dtype=torch.float32)
                X_macro= X_macro.to(DEVICE, dtype=torch.float32)
                y      = y.to(DEVICE,       dtype=torch.float32).view(-1, 1)

                preds = self.model(X_hf, X_macro)
                loss  = self.criterion(preds, y)

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            avg_train = train_loss / len(train_loader)

            # ── Validation ────────────────────────────────────────────
            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_hf, X_macro, y in val_loader:
                    X_hf    = X_hf.to(DEVICE,   dtype=torch.float32)
                    X_macro = X_macro.to(DEVICE, dtype=torch.float32)
                    y       = y.to(DEVICE,       dtype=torch.float32).view(-1, 1)
                    val_loss += self.criterion(self.model(X_hf, X_macro), y).item()

            avg_val = val_loss / len(val_loader)

            # [IMP-2] Step LR scheduler
            self.scheduler.step(avg_val)

            logger.info(
                f"Epoch {epoch+1:>4}/{epochs} | "
                f"Train: {avg_train:.5f} | Val: {avg_val:.5f} | "
                f"LR: {self.optimizer.param_groups[0]['lr']:.2e}"
            )

            self.history['train'].append(avg_train)
            self.history['val'].append(avg_val)

            # ── Checkpointing ─────────────────────────────────────────
            if avg_val < best_val_loss:
                best_val_loss    = avg_val
                patience_counter = 0
                torch.save(self.model.state_dict(), save_path)
                logger.info(f"  ✓ New best model saved → {save_path}")
            else:
                patience_counter += 1
                # [IMP-3] Early stopping
                if patience_counter >= self.es_patience:
                    logger.info(
                        f"Early stopping triggered after {epoch+1} epochs "
                        f"(no improvement for {self.es_patience} epochs)."
                    )
                    break

        logger.info(f"Training complete. Best val loss: {best_val_loss:.5f}")

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------
    def predict(self, loader: DataLoader) -> np.ndarray:
        """
        [FIX-4] Runs inference and returns predictions in real-world €/kg
        via inverse_transform on the target scaler.
        """
        self.model.eval()
        preds_scaled = []
        with torch.no_grad():
            for X_hf, X_macro, _ in loader:
                X_hf    = X_hf.to(DEVICE,   dtype=torch.float32)
                X_macro = X_macro.to(DEVICE, dtype=torch.float32)
                out = self.model(X_hf, X_macro).cpu().numpy()
                preds_scaled.append(out)

        preds_scaled = np.concatenate(preds_scaled, axis=0)
        return self.sequence_builder.inverse_transform_target(preds_scaled)


# ==============================================================================
# 5. FEATURE CATALOGUE (reference)
# ==============================================================================
#
# Target (Y):
#   aove_price_eur_kg      [Float32] : Weekly origin price (€/kg).
#
# High-Frequency Dynamic — LSTM sequence (spatially weighted average):
#   rainfall_mm            [Float32] : Accumulated weekly precipitation (mm).
#   temp_max_c             [Float32] : Absolute weekly maximum temperature (°C).
#   water_deficit_mm       [Float32] : P − ETP weekly balance (mm).
#   time_sin               [Float32] : sin(week × 2π / 52).
#   time_cos               [Float32] : cos(week × 2π / 52).
#
# Low-Frequency Macroeconomic — fused post-LSTM (converted to % delta / log):
#   stock_delta_pct        [Float32] : Olive oil stock variation (%).
#   surface_delta_pct      [Float32] : Productive surface area variation (%).
#   ipc_monthly            [Float32] : Consumer Price Index (inflation proxy).
#   diesel_price_eur       [Float32] : Agricultural diesel cost floor (€/L).
#   aove_lag_price         [Float32] : AOVE price lagged N weeks (explicit lag,
#                                      NOT the current target — leakage-free).


# ==============================================================================
# 6. EXPERIMENT VISUALISER
# ==============================================================================
class AOVEVisualiser:
    """
    Self-contained visualisation toolkit for post-training analysis.

    Generates four publication-ready matplotlib figures:
      1. Learning curve      — train vs val loss per epoch.
      2. Prediction vs actual— scatter + identity line on the val set.
      3. Residuals over time — signed error to spot temporal drift.
      4. Metrics dashboard   — MAE, RMSE, MAPE, R² as a compact table.

    Usage
    -----
    All methods are independent; call only what you need.

        vis = AOVEVisualiser(output_dir="plots")
        vis.learning_curve(train_losses, val_losses)
        vis.prediction_vs_actual(y_true, y_pred)
        vis.residuals_over_time(y_true, y_pred, dates)
        vis.metrics_dashboard(y_true, y_pred)

    Or run everything in one shot:

        vis.full_report(train_losses, val_losses, y_true, y_pred, dates)
    """

    def __init__(self, output_dir: str = "plots") -> None:
        try:
            import matplotlib.pyplot as plt
            import matplotlib.gridspec as gridspec
        except ImportError:
            raise ImportError("matplotlib is required for AOVEVisualiser. pip install matplotlib")

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self._style()

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _style() -> None:
        """Apply a clean, minimal style that renders well in notebooks and files."""
        import matplotlib.pyplot as plt
        plt.rcParams.update({
            "figure.facecolor":  "white",
            "axes.facecolor":    "white",
            "axes.edgecolor":    "#cccccc",
            "axes.grid":         True,
            "grid.color":        "#eeeeee",
            "grid.linewidth":    0.8,
            "axes.spines.top":   False,
            "axes.spines.right": False,
            "font.family":       "sans-serif",
            "font.size":         11,
            "axes.titlesize":    13,
            "axes.titleweight":  "bold",
            "axes.labelsize":    11,
            "legend.frameon":    False,
        })

    @staticmethod
    def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
        """
        Returns MAE, RMSE, MAPE, and R² for a pair of real-world arrays.
        MAPE excludes samples where |y_true| < 0.01 to avoid division by zero.
        """
        mae  = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

        mask = np.abs(y_true) >= 0.01
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

        return {"MAE (€/kg)": mae, "RMSE (€/kg)": rmse, "MAPE (%)": mape, "R²": r2}

    def _savefig(self, fig, name: str) -> None:
        import matplotlib.pyplot as plt
        path = os.path.join(self.output_dir, name)
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        logger.info(f"Plot saved → {path}")

    # ------------------------------------------------------------------
    # 1. Learning curve
    # ------------------------------------------------------------------

    def learning_curve(
        self,
        train_losses: list,
        val_losses:   list,
        filename: str = "01_learning_curve.png",
    ) -> None:
        """
        Plots HuberLoss per epoch for train and validation.
        A vertical dashed line marks the best val epoch (minimum val loss).
        """
        import matplotlib.pyplot as plt

        epochs     = range(1, len(train_losses) + 1)
        best_epoch = int(np.argmin(val_losses)) + 1

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(epochs, train_losses, label="Train loss",      color="#2563eb", linewidth=1.8)
        ax.plot(epochs, val_losses,   label="Validation loss", color="#dc2626", linewidth=1.8)
        ax.axvline(best_epoch, color="#16a34a", linestyle="--", linewidth=1.2,
                   label=f"Best epoch ({best_epoch})")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("HuberLoss (scaled space)")
        ax.set_title("Learning curve")
        ax.legend()
        fig.tight_layout()
        self._savefig(fig, filename)

    # ------------------------------------------------------------------
    # 2. Prediction vs actual
    # ------------------------------------------------------------------

    def prediction_vs_actual(
        self,
        y_true:   np.ndarray,
        y_pred:   np.ndarray,
        filename: str = "02_pred_vs_actual.png",
    ) -> None:
        """
        Scatter plot of predicted vs actual €/kg values on the validation set.
        The identity line (perfect prediction) is overlaid in green.
        Points are coloured by absolute error to highlight the worst predictions.
        """
        import matplotlib.pyplot as plt
        import matplotlib.cm as cm

        abs_err = np.abs(y_true - y_pred)
        vmax    = np.percentile(abs_err, 95)   # cap colour scale at p95 to avoid outlier dominance

        fig, ax = plt.subplots(figsize=(6, 6))
        sc = ax.scatter(y_true, y_pred, c=abs_err, cmap="YlOrRd",
                        vmin=0, vmax=vmax, alpha=0.75, edgecolors="none", s=30)
        lims = [min(y_true.min(), y_pred.min()) * 0.97,
                max(y_true.max(), y_pred.max()) * 1.03]
        ax.plot(lims, lims, color="#16a34a", linewidth=1.5, label="Perfect prediction")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("Actual price (€/kg)")
        ax.set_ylabel("Predicted price (€/kg)")
        ax.set_title("Prediction vs actual — validation set")
        ax.legend()
        cb = fig.colorbar(sc, ax=ax, shrink=0.8)
        cb.set_label("Absolute error (€/kg)")
        fig.tight_layout()
        self._savefig(fig, filename)

    # ------------------------------------------------------------------
    # 3. Residuals over time
    # ------------------------------------------------------------------

    def residuals_over_time(
        self,
        y_true:   np.ndarray,
        y_pred:   np.ndarray,
        dates:    pd.DatetimeIndex,
        filename: str = "03_residuals.png",
    ) -> None:
        """
        Plots signed residuals (pred − actual) over time.
        A zero-error reference line and a ±1 MAE band are shown.
        Temporal drift (systematic bias building up over time) is immediately
        visible if the residuals trend away from zero.
        """
        import matplotlib.pyplot as plt

        residuals = y_pred - y_true
        mae       = np.mean(np.abs(residuals))

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.fill_between(dates, -mae, mae, color="#2563eb", alpha=0.10, label="±MAE band")
        ax.plot(dates, residuals, color="#2563eb", linewidth=1.2, alpha=0.85)
        ax.axhline(0, color="#374151", linewidth=1.0, linestyle="--")
        ax.set_xlabel("Date")
        ax.set_ylabel("Residual (pred − actual, €/kg)")
        ax.set_title("Residuals over time — validation set")
        ax.legend()
        fig.tight_layout()
        self._savefig(fig, filename)

    # ------------------------------------------------------------------
    # 4. Metrics dashboard
    # ------------------------------------------------------------------

    def metrics_dashboard(
        self,
        y_true:   np.ndarray,
        y_pred:   np.ndarray,
        filename: str = "04_metrics_dashboard.png",
    ) -> None:
        """
        Renders MAE, RMSE, MAPE, and R² as large KPI cards in a single figure.
        Designed to be included directly in reports or slides.
        """
        import matplotlib.pyplot as plt

        metrics = self._compute_metrics(y_true, y_pred)
        labels  = list(metrics.keys())
        values  = list(metrics.values())
        fmts    = [".3f", ".3f", ".1f", ".4f"]

        fig, axes = plt.subplots(1, 4, figsize=(12, 3))
        colors = ["#2563eb", "#7c3aed", "#db2777", "#16a34a"]

        for ax, label, value, fmt, color in zip(axes, labels, values, fmts, colors):
            ax.set_facecolor(color + "18")          # light tinted background
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            ax.axis("off")
            ax.text(0.5, 0.62, f"{value:{fmt}}", ha="center", va="center",
                    fontsize=28, fontweight="bold", color=color,
                    transform=ax.transAxes)
            ax.text(0.5, 0.28, label, ha="center", va="center",
                    fontsize=12, color="#374151",
                    transform=ax.transAxes)
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(1.5)
                spine.set_visible(True)

        fig.suptitle("Model performance — validation set", fontsize=14, fontweight="bold", y=1.02)
        fig.tight_layout()
        self._savefig(fig, filename)
        logger.info("Metrics: " + " | ".join(f"{k}: {v:.4f}" for k, v in metrics.items()))

    # ------------------------------------------------------------------
    # 5. Full report (convenience wrapper)
    # ------------------------------------------------------------------

    def full_report(
        self,
        train_losses: list,
        val_losses:   list,
        y_true:       np.ndarray,
        y_pred:       np.ndarray,
        dates:        pd.DatetimeIndex,
    ) -> None:
        """
        Generates all four plots in one call and prints the metrics table to the log.
        """
        logger.info(f"Generating full visual report → {self.output_dir}/")
        self.learning_curve(train_losses, val_losses)
        self.prediction_vs_actual(y_true, y_pred)
        self.residuals_over_time(y_true, y_pred, dates)
        self.metrics_dashboard(y_true, y_pred)
        logger.info("Full report complete.")

# ==============================================================================
# 5b. FINE-TUNER
# ==============================================================================
class AOVEFineTuner:
    """
    Fine-tunes a pre-trained AOVEPricePredictor on recent data.

    Problem this solves
    -------------------
    The base model was trained on 2015-2023 data. The validation set
    (2024-2026) shows two failure modes visible in the diagnostics:
      1. Large negative residuals at the start of val (Jan-Mar 2024):
         the model underestimates the post-drought price spike because
         it never saw prices above ~4 EUR/kg during training.
      2. A ~3 EUR/kg positive spike around Jan 2025: a speculative shock
         the climate features cannot predict.

    Fine-tuning strategy
    --------------------
    Rather than retraining from scratch, we:
      1. Load the saved checkpoint (best_aove_model.pth).
      2. FREEZE the LSTM layers — they already encode 8 years of
         agroclimatic patterns correctly and we do not want to overwrite
         them with only 1-2 years of new data (catastrophic forgetting).
      3. UNFREEZE only the FC fusion head (fc.*) — this is where the
         price-level calibration happens and where the model needs to
         adapt to the new price regime (4-9 EUR/kg vs 1-3 EUR/kg).
      4. Train with a very low LR (1e-4 default) on a recent window
         of data (last N weeks, configurable via --fine_window_weeks).
      5. Save to a separate checkpoint (finetuned_aove_model.pth) so
         the original model is never overwritten.

    What layers are frozen vs unfrozen
    ------------------------------------
      FROZEN  : lstm.*          (captures agroclimatic seasonality)
      FROZEN  : dropout (implicit, part of LSTM)
      UNFROZEN: fc.0  Linear(fusion_dim, 64)
      UNFROZEN: fc.1  ReLU
      UNFROZEN: fc.2  Dropout
      UNFROZEN: fc.3  Linear(64, 32)
      UNFROZEN: fc.4  ReLU
      UNFROZEN: fc.5  Linear(32, 1)

    Usage
    -----
    Called automatically when --finetune is passed to __main__.
    Can also be used standalone:

        finetuner = AOVEFineTuner(
            model=model,
            sequence_builder=builder,
            checkpoint_path="best_aove_model.pth",
        )
        finetuner.load_and_freeze()
        finetuner.finetune(
            train_loader=ft_loader,
            val_loader=val_loader,
            epochs=30,
            save_path="finetuned_aove_model.pth",
        )
    """

    def __init__(
        self,
        model: nn.Module,
        sequence_builder: "TemporalSequenceBuilder",
        checkpoint_path: str,
        learning_rate: float = 1e-4,
        lr_patience: int = 5,
        es_patience: int = 10,
    ) -> None:
        self.model             = model.to(DEVICE)
        self.sequence_builder  = sequence_builder
        self.checkpoint_path   = checkpoint_path
        self.criterion         = nn.HuberLoss(delta=1.0)
        self.lr                = learning_rate
        self.lr_patience       = lr_patience
        self.es_patience       = es_patience
        self.history: dict     = {"train": [], "val": []}

    def load_and_freeze(self) -> None:
        """
        Loads the checkpoint and freezes all LSTM parameters.
        Only the FC head remains trainable.
        """
        if not os.path.exists(self.checkpoint_path):
            raise FileNotFoundError(
                f"Checkpoint not found: {self.checkpoint_path}\n"
                "Train the base model first (run without --finetune)."
            )
        self.model.load_state_dict(
            torch.load(self.checkpoint_path, map_location=DEVICE)
        )
        logger.info(f"Checkpoint loaded from {self.checkpoint_path}")

        # Freeze LSTM layers
        for name, param in self.model.named_parameters():
            if name.startswith("lstm"):
                param.requires_grad = False

        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in self.model.parameters() if not p.requires_grad)
        logger.info(
            f"Layer freeze complete: "
            f"{frozen:,} params frozen (LSTM) | "
            f"{trainable:,} params trainable (FC head)"
        )

    def finetune(
        self,
        train_loader: DataLoader,
        val_loader:   DataLoader,
        epochs: int = 30,
        save_path: str = "finetuned_aove_model.pth",
    ) -> None:
        """
        Fine-tunes only the unfrozen FC head parameters.
        Uses a separate optimizer so LR does not interfere with base training.
        """
        # Only pass trainable parameters to the optimizer
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.lr,
            weight_decay=1e-5,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=self.lr_patience, factor=0.5
        )

        best_val_loss    = float("inf")
        patience_counter = 0

        logger.info(f"Fine-tuning FC head for up to {epochs} epochs (LR={self.lr:.1e})...")

        for epoch in range(epochs):
            # ── Training ──────────────────────────────────────────────────
            self.model.train()
            train_loss = 0.0
            for X_hf, X_macro, y in train_loader:
                X_hf    = X_hf.to(DEVICE,   dtype=torch.float32)
                X_macro = X_macro.to(DEVICE, dtype=torch.float32)
                y       = y.to(DEVICE,       dtype=torch.float32).view(-1, 1)

                preds = self.model(X_hf, X_macro)
                loss  = self.criterion(preds, y)

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()

            avg_train = train_loss / len(train_loader)

            # ── Validation ────────────────────────────────────────────────
            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_hf, X_macro, y in val_loader:
                    X_hf    = X_hf.to(DEVICE,   dtype=torch.float32)
                    X_macro = X_macro.to(DEVICE, dtype=torch.float32)
                    y       = y.to(DEVICE,       dtype=torch.float32).view(-1, 1)
                    val_loss += self.criterion(self.model(X_hf, X_macro), y).item()

            avg_val = val_loss / len(val_loader)
            scheduler.step(avg_val)
            self.history["train"].append(avg_train)
            self.history["val"].append(avg_val)

            logger.info(
                f"FT Epoch {epoch+1:>3}/{epochs} | "
                f"Train: {avg_train:.5f} | Val: {avg_val:.5f} | "
                f"LR: {optimizer.param_groups[0]['lr']:.2e}"
            )

            if avg_val < best_val_loss:
                best_val_loss    = avg_val
                patience_counter = 0
                torch.save(self.model.state_dict(), save_path)
                logger.info(f"  ✓ Best fine-tuned model saved -> {save_path}")
            else:
                patience_counter += 1
                if patience_counter >= self.es_patience:
                    logger.info(
                        f"Fine-tune early stopping at epoch {epoch+1} "
                        f"(no improvement for {self.es_patience} epochs)."
                    )
                    break

        logger.info(f"Fine-tuning complete. Best val loss: {best_val_loss:.5f}")

    def predict(self, loader: DataLoader) -> np.ndarray:
        """Returns fine-tuned predictions in real-world EUR/kg."""
        self.model.eval()
        preds_scaled = []
        with torch.no_grad():
            for X_hf, X_macro, _ in loader:
                X_hf    = X_hf.to(DEVICE,   dtype=torch.float32)
                X_macro = X_macro.to(DEVICE, dtype=torch.float32)
                out = self.model(X_hf, X_macro).cpu().numpy()
                preds_scaled.append(out)
        preds_scaled = np.concatenate(preds_scaled, axis=0)
        return self.sequence_builder.inverse_transform_target(preds_scaled)

# ==============================================================================
# ENTRY POINT
# ==============================================================================
def build_dataloaders(
    splits: dict,
    batch_size: int,
) -> Tuple[DataLoader, DataLoader]:
    """Converts numpy arrays to TensorDatasets and wraps them in DataLoaders."""
    def to_loader(arrays, shuffle):
        tensors = [torch.from_numpy(a) for a in arrays]
        return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)

    train_loader = to_loader(splits["train"], shuffle=True)
    val_loader   = to_loader(splits["val"],   shuffle=False)
    return train_loader, val_loader

if __name__ == "__main__":
    DEFAULT_CLIMATE = "./data/climate_dataset.csv"
    DEFAULT_MACRO   = "./data/macro_dataset.csv"

    parser = argparse.ArgumentParser(description="AOVE Macro-Aware LSTM Production Engine v2")
    parser.add_argument("--climate_csv",  type=str, default=DEFAULT_CLIMATE, help="Municipal climate CSV path")
    parser.add_argument("--macro_csv",    type=str, default=DEFAULT_MACRO,   help="Macroeconomic CSV path")
    parser.add_argument("--time_steps",   type=int, default=104,    help="Sliding window length (weeks)")
    parser.add_argument("--epochs",       type=int, default=200,    help="Maximum training epochs")
    parser.add_argument("--batch_size",   type=int, default=32,     help="Batch size")
    parser.add_argument("--train_ratio",  type=float, default=0.8,  help="Chronological train fraction")
    parser.add_argument("--lr",           type=float, default=5e-4, help="Initial learning rate")
    parser.add_argument("--lr_patience",  type=int, default=10,     help="LR scheduler patience (epochs)")
    parser.add_argument("--es_patience",  type=int, default=20,     help="Early stopping patience (epochs)")
    parser.add_argument("--aove_lag",     type=int, default=1,      help="AOVE price lag weeks (0 = disable)")
    parser.add_argument("--save_path",    type=str, default="best_aove_model.pth", help="Checkpoint path")
    # Fine-tuning arguments
    parser.add_argument("--finetune",        action="store_true", default=False,
                        help="Fine-tune the FC head of a pre-trained model on recent data.")
    parser.add_argument("--fine_checkpoint",   type=str, default="best_aove_model.pth",
                        help="Checkpoint to load for fine-tuning (default: best_aove_model.pth).")
    parser.add_argument("--fine_window_weeks", type=int, default=220,
                        help="Number of most-recent weeks to use as fine-tune training set (default: 220 = ~4 years).")
    parser.add_argument("--fine_epochs",       type=int, default=15,
                        help="Maximum fine-tuning epochs (default: 15).")
    parser.add_argument("--fine_lr",           type=float, default=1e-4,
                        help="Fine-tuning learning rate (default: 1e-4, lower than base 1e-3).")
    parser.add_argument("--fine_save_path",    type=str, default="finetuned_aove_model.pth",
                        help="Output path for the fine-tuned checkpoint.")
    
# ==============================================================================
# HYPERPARAMETER DICTIONARY & CONFIGURATION REFERENCE
# ==============================================================================
#
# BASE TRAINING PARAMETERS
# ------------------------
# --climate_csv   : Path to the municipal climate dataset (spatially aggregated later).
# --macro_csv     : Path to the macroeconomic indicators dataset (IPC, diesel, stocks).
# --time_steps    : Lookback window size in weeks. Defines the temporal receptive field 
#                   of the LSTM (e.g., 104 weeks captures the biennial bearing cycle of olive trees).
# --epochs        : Absolute upper bound for training iterations over the entire dataset. 
#                   Usually preempted by early stopping to prevent memorization.
# --batch_size    : Number of sequence windows processed concurrently in a single forward/backward pass. 
#                   Smaller batches (e.g., 32) can act as a regularizer and speed up convergence.
# --train_ratio   : The proportion of the dataset allocated to the training split, strictly 
#                   enforced chronologically to prevent future data leakage.
# --lr            : Base learning rate for the Adam optimizer during initial full-model training. 
#                   Controls the initial step size for gradient descent (standard: 1e-3).
# --lr_patience   : Number of epochs with no improvement in validation loss before the 
#                   ReduceLROnPlateau scheduler halves the learning rate.
# --es_patience   : Early stopping patience. Halts training if the validation loss stagnates 
#                   for this many consecutive epochs, mitigating catastrophic overfitting.
# --aove_lag      : Number of weeks to delay the target variable (AOVE price) before injecting it 
#                   as an autoregressive feature, mimicking real-world reporting latency.
# --save_path     : Filepath where the best base model state dictionary (weights) will be serialized.
#
# TRANSFER LEARNING & FINE-TUNING PARAMETERS
# ------------------------------------------
# --finetune          : Boolean flag triggering the Transfer Learning mode. Freezes the LSTM layers 
#                       to preserve learned climatic patterns while retraining the Fully Connected head.
# --fine_checkpoint   : Path to the pre-trained base model weights to be loaded prior to fine-tuning.
# --fine_window_weeks : The subset of the most recent training weeks used exclusively for fine-tuning. 
#                       Allows the FC head to calibrate to the latest inflationary market regime.
# --fine_epochs       : Maximum number of training iterations allocated strictly for the fine-tuning phase.
# --fine_lr           : A deliberately lower learning rate (e.g., 1e-4) used during fine-tuning to ensure 
#                       granular weight updates without destroying pre-learned representations.
# --fine_save_path    : Destination filepath for the newly fine-tuned model checkpoint, preserving 
#                       the original base model untouched.
# ==============================================================================

    # Strip Jupyter/VS Code injected flags before argparse sees them
    import sys
    sys.argv = [sys.argv[0]] + [
        a for a in sys.argv[1:]
        if not a.startswith('--f=') and not a.startswith('-f=')
    ]
    args, _ = parser.parse_known_args()

    logger.info("=== AOVE Deep Learning Production Engine ===")
    logger.info(f"Device: {DEVICE}")
    logger.info(f"Using Climate: {args.climate_csv} | Macro: {args.macro_csv}")

    # ── 1. Load raw CSVs ──────────────────────────────────────────────────────
    df_municipal = pd.read_csv(args.climate_csv, parse_dates=['date'])
    df_macro     = pd.read_csv(args.macro_csv,   parse_dates=['reference_date'])

    # ── 2. ETL ────────────────────────────────────────────────────────────────
    etl = MarketETLPipeline(publish_delay_days=15)
    df_climate = etl.aggregate_climate(df_municipal)
    df_final   = etl.align_macro_data(df_climate, df_macro, aove_lag_weeks=args.aove_lag)

    # ── 3. Define feature sets ────────────────────────────────────────────────
    HF_COLS = ['rainfall_mm', 'temp_max_c', 'water_deficit_mm', 'time_sin', 'time_cos']

    MACRO_COLS = ['stock_delta_pct', 'surface_delta_pct', 'ipc_monthly', 'diesel_price_eur']
    if args.aove_lag > 0:
        MACRO_COLS.append('aove_lag_price')

    TARGET_COL = 'aove_price_eur_kg'

    # ── 4. Build sequences ────────────────────────────────────────────────────
    builder = TemporalSequenceBuilder(time_steps=args.time_steps)
    splits  = builder.build_sequences_split(
        df_final, HF_COLS, MACRO_COLS, TARGET_COL, train_ratio=args.train_ratio
    )

    train_loader, val_loader = build_dataloaders(splits, args.batch_size)

    # ── 5. Instantiate model ──────────────────────────────────────────────────
    model = AOVEPricePredictor(
        hf_input_dim=len(HF_COLS),
        macro_input_dim=len(MACRO_COLS),
    )
    logger.info(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # ── 6. Train or Fine-tune ─────────────────────────────────────────────────
    if not args.finetune:
        # ── Standard full training ────────────────────────────────────────────
        trainer = AOVETrainer(
            model=model,
            sequence_builder=builder,
            learning_rate=args.lr,
            lr_patience=args.lr_patience,
            es_patience=args.es_patience,
        )
        trainer.train_and_validate(
            train_loader, val_loader,
            epochs=args.epochs,
            save_path=args.save_path,
        )
        active_trainer = trainer

    else:
        # ── Fine-tuning mode ──────────────────────────────────────────────────
        logger.info("=== FINE-TUNING MODE ===")
        logger.info(
            f"FC head will be fine-tuned on the last {args.fine_window_weeks} weeks. "
            f"LSTM layers frozen. LR={args.fine_lr:.1e}"
        )

        # Build a fine-tune DataLoader from the most recent ft_window_weeks
        # of the TRAINING set (chronologically latest, to capture new price regime)
        X_hf_tr, X_macro_tr, y_tr = splits["train"]
        ft_start = max(0, len(X_hf_tr) - args.fine_window_weeks)
        ft_splits = {
            "train": (X_hf_tr[ft_start:], X_macro_tr[ft_start:], y_tr[ft_start:]),
            "val":   splits["val"],
        }
        ft_train_loader, ft_val_loader = build_dataloaders(ft_splits, args.batch_size)

        logger.info(
            f"Fine-tune set: {len(ft_splits['train'][0])} windows "
            f"(last {args.fine_window_weeks} of training)"
        )

        finetuner = AOVEFineTuner(
            model=model,
            sequence_builder=builder,
            checkpoint_path=args.fine_checkpoint,
            learning_rate=args.fine_lr,
            lr_patience=5,
            es_patience=5,
        )
        finetuner.load_and_freeze()

        # Increase FC dropout to 0.5 before fine-tuning (regularisation boost)
        for module in finetuner.model.fc.modules():
            if isinstance(module, __import__('torch').nn.Dropout):
                module.p = 0.5

        finetuner.finetune(
            train_loader=ft_train_loader,
            val_loader=ft_val_loader,
            epochs=args.fine_epochs,
            save_path=args.fine_save_path,
        )
        active_trainer = finetuner

    # ── 7. Validation predictions in €/kg ────────────────────────────────────
    val_preds_eur = active_trainer.predict(val_loader)
    logger.info(f"Sample val predictions (€/kg): {val_preds_eur[:5]}")

    # ── 8. Visual report ──────────────────────────────────────────────────────
    logger.info("Initializing visual diagnostics...")
    output_tag = "finetuned" if args.finetune else "base"
    y_true_eur = builder.inverse_transform_target(splits['val'][2])
    visualiser = AOVEVisualiser(output_dir=f"aove_diagnostics_{output_tag}")
    visualiser.full_report(
        train_losses=active_trainer.history['train'],
        val_losses=active_trainer.history['val'],
        y_true=y_true_eur,
        y_pred=val_preds_eur,
        dates=splits['val_dates'],
    )
    logger.info("Pipeline execution finished successfully.")

INFO: === AOVE Deep Learning Production Engine ===
INFO: Device: cuda
INFO: Using Climate: ./data/climate_dataset.csv | Macro: ./data/macro_dataset.csv
INFO: Performing spatial weighted aggregation on climate data...
INFO: Applying 15-day publication shift to macro data...
INFO: Building sequences with chronological train/val split...
INFO: Total windows: 749 | Train: 578 | Val: 171
INFO: Model parameters: 211,905
INFO: Epoch    1/200 | Train: 0.25299 | Val: 3.75040 | LR: 5.00e-04
INFO:   ✓ New best model saved → best_aove_model.pth
INFO: Epoch    2/200 | Train: 0.22133 | Val: 3.42542 | LR: 5.00e-04
INFO:   ✓ New best model saved → best_aove_model.pth
INFO: Epoch    3/200 | Train: 0.17747 | Val: 2.80840 | LR: 5.00e-04
INFO:   ✓ New best model saved → best_aove_model.pth
INFO: Epoch    4/200 | Train: 0.14283 | Val: 2.45271 | LR: 5.00e-04
INFO:   ✓ New best model saved → best_aove_model.pth
INFO: Epoch    5/200 | Train: 0.11382 | Val: 2.11192 | LR: 5.00e-04
INFO:   ✓ New best model saved